In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import numpy as np
import pandas as pd


def stratified_downsample_pkl(
    in_pkl: str,
    out_pkl: str,
    out_csv: str | None = None,
    *,
    target_n: int,
    n_len_bins: int = 5,
    n_cons_bins: int = 5,
    seed: int = 42,
    input_col: str = "input",
    constraints_col: str = "constraints",
) -> pd.DataFrame:
    df = pd.read_pickle(in_pkl)
    original_cols = df.columns.tolist()

    if target_n > len(df):
        raise ValueError(f"target_n={target_n} > dataset size={len(df)}")
    if input_col not in df.columns or constraints_col not in df.columns:
        raise ValueError(f"PKL must contain '{input_col}' and '{constraints_col}'")

    # --- getting length + constraint count (works when input is list[dict])
    def msg_text_len(x) -> int:
        if isinstance(x, list):
            total = 0
            for m in x:
                if isinstance(m, dict):
                    c = m.get("content", "")
                    if isinstance(c, str):
                        total += len(c)
                elif isinstance(m, str):
                    total += len(m)
            return total
        if isinstance(x, dict):
            c = x.get("content", "")
            return len(c) if isinstance(c, str) else len(str(c))
        if isinstance(x, str):
            return len(x)
        return len(str(x))

    def constraint_count(x) -> int:
        if isinstance(x, list):
            return len(x)
        if isinstance(x, dict):
            # common: {"evaluation": [...], ...}
            for v in x.values():
                if isinstance(v, list):
                    return len(v)
            return 1
        if isinstance(x, str):
            return x.count("'desc':") + x.count('"desc":')
        return 0

    length_proxy = df[input_col].apply(msg_text_len)
    cons_count = df[constraints_col].apply(constraint_count)

    # --- bucketing each row by length + constraints (quantiles)
    len_bin = pd.qcut(length_proxy.rank(method="average"), q=n_len_bins, labels=False, duplicates="drop")
    cons_bin = pd.qcut((cons_count + 1).rank(method="average"), q=n_cons_bins, labels=False, duplicates="drop")

    work = df.copy()
    work["_key"] = len_bin.astype(str) + "_" + cons_bin.astype(str)

    # --- computing how many to sample per stratum (proportional, exact total)
    sizes = work["_key"].value_counts().sort_index()
    ideal = (sizes / sizes.sum()) * target_n

    base = pd.Series(np.floor(ideal.to_numpy()).astype(int), index=sizes.index)
    frac = ideal - base
    base = np.minimum(base, sizes)

    remaining = target_n - int(base.sum())
    if remaining > 0:
        for k in frac.sort_values(ascending=False).index:
            if remaining == 0:
                break
            if base.loc[k] < sizes.loc[k]:
                base.loc[k] += 1
                remaining -= 1

    if int(base.sum()) != target_n:
        raise RuntimeError(f"Allocation failed: expected {target_n}, got {int(base.sum())}")

    # --- sampling within each stratum
    rng = np.random.RandomState(seed)
    parts = []
    for k, n in base.items():
        if n <= 0:
            continue
        g = work[work["_key"] == k]
        parts.append(g.sample(n=int(n), random_state=rng) if len(g) > n else g)

    sampled = pd.concat(parts, axis=0).sort_index()
    sampled = sampled[original_cols]

    # --- write outputs
    sampled.to_pickle(out_pkl)
    if out_csv:
        sampled.to_csv(out_csv, index=False)

    return sampled


In [ ]:
 if __name__ == "__main__":
    name_string = "/content/drive/MyDrive/TokenReductionProject/Data/eval_criteria_with_attention_scores_probe_Answer: _model_Qwen2.5-7B_"
    in_pkl = name_string+".pkl"
    out_csv = name_string+"DOWNSAMPLED.csv"
    out_pkl = name_string+"DOWNSAMPLED.pkl"

    stratified_downsample_pkl(
        in_pkl,
        out_pkl,
        out_csv,
        target_n=128,     # change to 707//5 (~141) if you want exactly 1/5
        n_len_bins=5,
        n_cons_bins=5,
        seed=42,
    )
    print("Done.")

In [ ]:
def remove_column_from_pkl(
    in_pkl: str,
    column_to_remove: str,
    out_pkl: str,
    out_csv: str | None = None,
) -> pd.DataFrame:
    df = pd.read_pickle(in_pkl)

    if column_to_remove not in df.columns:
        raise ValueError(f"Column '{column_to_remove}' not found. Available columns: {list(df.columns)}")

    df = df.drop(columns=[column_to_remove])

    df.to_pickle(out_pkl)

    if out_csv:
        df.to_csv(out_csv, index=False)

    return df

In [ ]:
new_df = remove_column_from_pkl(
    in_pkl=name_string+"DOWNSAMPLED.pkl",
    column_to_remove="scored_spans",
    out_pkl="/content/drive/MyDrive/TokenReductionProject/Data/eval_criteria_DOWNSAMPLED.pkl",
    out_csv="/content/drive/MyDrive/TokenReductionProject/Data/eval_criteria_DOWNSAMPLED.csv",
)

print(new_df.columns)
